[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/notebooks/m5_harness.ipynb) [![Course](https://img.shields.io/badge/Full_Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-5)

# Module 5 — Harness Architecture

**Level:** Advanced | **Time:** ~2h | [Full reading →](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-5)

### What you'll build
A production-grade agent harness with Initializer/Executor pattern, replay log, and circuit breaker.

### Key concepts
- **Two-agent pattern**: Initializer decomposes → feature list; Executor picks next failing feature, implements, verifies
- **Structured feature list**: `{name, steps: [], status: failing|passing|skipped}` — forces verification before marking done
- **Git as external memory**: one feature per commit, descriptive messages, progress notes per session
- **Replay log**: store `(step_id, prompt_hash, response, tool_calls, ts)` for deterministic replay of failures
- **Circuit breaker**: same tool + same args N times → force different action; prevents infinite loops
- **Step budget**: hard cap with escalation on exceed

### Failure modes

| Failure | Root cause | Fix |
|---------|-----------|-----|
| Premature victory | No verification gate | Self-test before status = passing |
| Undocumented progress | No commit discipline | Require git commit + progress note per feature |
| Infinite loop | Ambiguous completion | Circuit breaker + step budget |

---

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Option A: OpenAI API (recommended for Colab)
!pip install openai --quiet

import os
from openai import OpenAI

# Set your OpenAI API key — in Colab: Secrets (🔑) → add OPENAI_API_KEY
# Then enable notebook access, or paste directly (don't commit keys to git)
# os.environ["OPENAI_API_KEY"] = "sk-..."   # ← uncomment and paste if not using Secrets

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o"

# ── Option B: Ollama (local, no API key, no cost) ─────────────────────────────
# 1. Install Ollama: https://ollama.com/download
# 2. Run: ollama pull llama3.2
# 3. Uncomment the two lines below and comment out the OpenAI lines above:
#
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# MODEL = "llama3.2"   # or: mistral, phi4, gemma3, qwen2.5, etc.
#
# Everything in this notebook works with either client — MODEL is passed through.
print(f"Client ready. Using model: {MODEL}")

In [ ]:
import os, json, time, hashlib
from dataclasses import dataclass, field
from typing import Optional
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ── Data structures ───────────────────────────────────────────────────────────

@dataclass
class Feature:
    name: str
    steps: list[str]
    status: str = "failing"   # failing | passing | skipped

@dataclass
class ReplayStep:
    step_id: int
    prompt_hash: str
    response_summary: str
    tool_calls: list[dict]
    timestamp: float

# ── Harness ───────────────────────────────────────────────────────────────────

class Harness:
    """
    Two-agent harness: Initializer decomposes task → feature list.
    Executor picks next failing feature, implements, self-verifies.
    """

    MAX_STEPS_PER_FEATURE = 15
    CIRCUIT_BREAKER_THRESHOLD = 3  # same tool+args N times → force different action

    def __init__(self, task: str, work_dir: str = "/tmp/harness-demo"):
        self.task = task
        self.work_dir = work_dir
        self.features: list[Feature] = []
        self.progress_log: list[str] = []
        self.replay_log: list[ReplayStep] = []
        self._step_counter = 0
        self._recent_tool_calls: list[str] = []  # for circuit breaker

    # ── Initializer ───────────────────────────────────────────────────────────

    def initialize(self) -> list[Feature]:
        """Decompose task into verifiable features."""
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": (
                    "You are a software project initializer. "
                    "Decompose the task into 3-5 concrete, verifiable features. "
                    "Return JSON: {features: [{name, steps: [str], status: \"failing\"}]}"
                )},
                {"role": "user", "content": f"Task: {self.task}"}
            ],
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
        self.features = [Feature(**f) for f in data["features"]]
        self._log_progress(f"Initialized: {len(self.features)} features decomposed from task.")
        return self.features

    # ── Executor ──────────────────────────────────────────────────────────────

    def execute_next(self) -> Optional[Feature]:
        """Pick next failing feature and attempt to implement it."""
        failing = [f for f in self.features if f.status == "failing"]
        if not failing:
            return None
        feature = failing[0]
        self._log_progress(f"Starting feature: {feature.name}")
        success = self._implement_feature(feature)
        feature.status = "passing" if success else "failing"
        self._log_progress(f"Feature '{feature.name}': {'PASSING' if success else 'STILL FAILING'}")
        return feature

    def _implement_feature(self, feature: Feature) -> bool:
        self._recent_tool_calls = []
        messages = [
            {"role": "system", "content": (
                "You are an executor agent. Implement the given feature step by step. "
                "After implementation, verify it works. "
                "Return JSON: {done: bool, verification: str, notes: str}"
            )},
            {"role": "user", "content": (
                f"Feature: {feature.name}\n"
                f"Steps: {json.dumps(feature.steps, indent=2)}\n"
                f"Progress log: {self._get_recent_progress()}"
            )}
        ]

        for step in range(self.MAX_STEPS_PER_FEATURE):
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content
            self._record_replay(messages[-1]["content"], content, [])
            result = json.loads(content)
            if result.get("done"):
                return True

        return False  # exhausted step budget

    # ── Circuit breaker ───────────────────────────────────────────────────────

    def _check_circuit_breaker(self, tool_name: str, args: dict) -> bool:
        """Return True if circuit should break (same call repeated N times)."""
        key = f"{tool_name}:{json.dumps(args, sort_keys=True)}"
        self._recent_tool_calls.append(key)
        recent = self._recent_tool_calls[-self.CIRCUIT_BREAKER_THRESHOLD:]
        return len(recent) >= self.CIRCUIT_BREAKER_THRESHOLD and len(set(recent)) == 1

    # ── Logging helpers ───────────────────────────────────────────────────────

    def _log_progress(self, message: str):
        entry = f"[{time.strftime('%H:%M:%S')}] {message}"
        self.progress_log.append(entry)
        print(entry)

    def _get_recent_progress(self) -> str:
        return "\n".join(self.progress_log[-5:])

    def _record_replay(self, prompt: str, response: str, tool_calls: list):
        self.replay_log.append(ReplayStep(
            step_id=self._step_counter,
            prompt_hash=hashlib.sha256(prompt.encode()).hexdigest()[:8],
            response_summary=response[:100],
            tool_calls=tool_calls,
            timestamp=time.time()
        ))
        self._step_counter += 1

    # ── Status report ─────────────────────────────────────────────────────────

    def status(self) -> dict:
        total = len(self.features)
        passing = sum(1 for f in self.features if f.status == "passing")
        return {
            "total": total, "passing": passing, "failing": total - passing,
            "features": [{"name": f.name, "status": f.status} for f in self.features],
            "replay_steps": len(self.replay_log)
        }


# ── Demo ──────────────────────────────────────────────────────────────────────

harness = Harness(task="Build a URL shortener with click tracking")
print("=== Initializer ===")
features = harness.initialize()
for f in features:
    print(f"  - {f.name}: {f.steps[:1]}...")

print("\n=== Executor (1 feature) ===")
result = harness.execute_next()

print("\n=== Status ===")
print(json.dumps(harness.status(), indent=2))

## Exercise

Extend the harness with parallel execution and rollback.

> **Interview question:** *"How do you build an agent that works across multiple context windows without losing state?"*

In [ ]:
# Exercise: Extend the harness with parallel execution + rollback
#
# 1. Add parallel_execute(features: list[Feature]) that runs independent
#    features concurrently using ThreadPoolExecutor
# 2. Add rollback_last() that reverts the last committed feature to "failing"
#    (simulate with in-memory state since we have no real git here)
# 3. Add a step_budget_exceeded callback that fires when MAX_STEPS_PER_FEATURE
#    is hit — should escalate to human or skip

from concurrent.futures import ThreadPoolExecutor

def parallel_execute(harness: Harness, max_workers: int = 3) -> list[Feature]:
    """Run independent failing features in parallel."""
    failing = [f for f in harness.features if f.status == "failing"]
    # TODO: implement fan-out/fan-in
    raise NotImplementedError

def rollback_last(harness: Harness) -> Optional[Feature]:
    """Revert last passing feature to failing."""
    # TODO: implement
    raise NotImplementedError